# Professionally You

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from PyPDF2 import PdfReader
import gradio as gr

c:\Users\thiru\Desktop\Engineering\AI Engineering\Projects\Course Projects\agentic AI\AgenticAI-Experiments\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
load_dotenv(override=True)
OLLAMA_MODEL = "gpt-oss:120b-cloud"
ollama_client = OpenAI(base_url="http://localhost:11434", api_key="ollama")
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [4]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [5]:
def push(message):
    print(f"Sending push notification: {message}")
    payload = {
        "token": pushover_token,
        "user": pushover_user,
        "message": message
    }
    response = requests.post(pushover_url, data=payload)
    if response.status_code == 200:
        print("Push notification sent successfully.")
    else:
        print(f"Failed to send push notification: {response.text}") 
    

In [7]:
push("Hi there! This is a test notification from your agentic AI system. Everything is set up and ready to go!")

Sending push notification: Hi there! This is a test notification from your agentic AI system. Everything is set up and ready to go!
Push notification sent successfully.


In [8]:
def record_user_details(email,name="not provided", notes = "not provided"):
    push(f"New user details recorded:\nEmail: {email}\nName: {name}\nNotes: {notes}")
    return {"recorded":"ok"}

In [9]:
def record_unknown_question(question):
    push(f"Received an unknown question:\n{question}")
    return {"recorded":"ok"}

In [31]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Record user details such as email, name, and notes.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of the user"
            },
            "name": {
                "type": "string",
                "description": "The name of the user"
            },
            "notes": {
                "type": "string",
                "description": "Additional notes about the user"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [32]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Record an unknown question that the agent could not answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The unknown question received by the agent"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [33]:
tools = [
    {"type": "function", "function":  record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

In [34]:
def handle_tool_calls(tool_calls):
    result = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"tool call received: {tool_name} with arguments {arguments}")

        if tool_name == "record_user_details":
            res = record_user_details(**arguments)
            result.append(res)
        elif tool_name == "record_unknown_question":
            res = record_unknown_question(**arguments)
            result.append(res)
    return result


In [15]:
globals()["record_unknown_question"]("This is really hard question")

Sending push notification: Received an unknown question:
This is really hard question
Push notification sent successfully.


{'recorded': 'ok'}

In [35]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"tool call received: {tool_name} with arguments {arguments}")

        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role":"tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [36]:
reader = PdfReader("../me/Profile.pdf")
linkedin = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text + "\n"
with open("../me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Jegan"

In [37]:
system_prompt = f"""
You are acting as {name}. you answering questions on {name}'s website\
particularly questions related to {name}'s professional background, experience, skills, and interests.\
You responsiblity is to represent {name} in the most accurate and professional way possible.\
You are given the following information to help you answer questions:
1. A summary of {name}'s professional background and experience:
{summary}
2. The content of {name}'s LinkedIn profile:
{linkedin}
Be professional, concise, and accurate in your responses. If you receive a question that you cannot answer based on the provided information, please use the 'record_unknown_question' tool to log the question for further review. Always strive to provide helpful and informative answers that reflect {name}'s expertise and experience.
If you dont know the answer to any queston use your record_unknown_question tool to record the question and respond with "I'm sorry, but I don't have the information to answer that question at the moment."
if the user is engaing in discussions try to steer them to towards geting in touch via email. ask for their email and any details you can get and use the record_user_details tool to record their information and respond with "Thank you for your interest! I've recorded your details and will get back to you as soon as possible."
"""

system_prompt += f"with this context plese chat with the user always staying in character as {name} and only answer questions related to {name}'s professional background, experience, skills, and interests. never break character or talk about anything unrelated to {name}'s professional profile."


In [38]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = openai_client.chat.completions.create(model="gpt-4o", messages=messages, tools=tools)
        
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


tool call received: record_user_details with arguments {'email': 'abs @ email.com'}
Sending push notification: New user details recorded:
Email: abs @ email.com
Name: not provided
Notes: not provided
Push notification sent successfully.
tool call received: record_unknown_question with arguments {'question': 'Who is the president of Sri Lanka in 2028?'}
Sending push notification: Received an unknown question:
Who is the president of Sri Lanka in 2028?
Push notification sent successfully.
